## 1 Скачивание датасета и предобработка данных


In [ ]:
import pandas as pd
# import kagglehub
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

import tensorflow as tf
from tensorflow import keras


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/data/training.csv')
# df = pd.read_csv('/data/training.csv')
df

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72978,73010,1,12/2/2009,ADESA,2001,8,MERCURY,SABLE,GS,4D SEDAN GS,...,4836.0,5937.0,NaN,NaN,18111,30212,GA,4200.0,0,993
72979,73011,0,12/2/2009,ADESA,2007,2,CHEVROLET,MALIBU 4C,LS,4D SEDAN LS,...,10151.0,11652.0,NaN,NaN,18881,30212,GA,6200.0,0,1038
72980,73012,0,12/2/2009,ADESA,2005,4,JEEP,GRAND CHEROKEE 2WD V,Lar,4D WAGON LAREDO,...,11831.0,14402.0,NaN,NaN,18111,30212,GA,8200.0,0,1893
72981,73013,0,12/2/2009,ADESA,2006,3,CHEVROLET,IMPALA,LS,4D SEDAN LS,...,10099.0,11228.0,NaN,NaN,18881,30212,GA,7000.0,0,1974


In [ ]:
# 1. Конвертируем PurchDate в datetime 
df['PurchDate'] = pd.to_datetime(df['PurchDate'], errors='coerce')

# 2. Сортируем по дате 
df = df.sort_values('PurchDate').reset_index(drop=True)

# 3. Вычисляем границы для 33% / 33% / 33%
n = len(df)
train_end = n // 3
valid_end = 2 * n // 3

# 4. Разделяем на три набора
train = df.iloc[:train_end].copy()
valid = df.iloc[train_end:valid_end].copy()
test = df.iloc[valid_end:].copy()

train

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,32389,0,2009-01-05,MANHEIM,2007,2,CHRYSLER,PACIFICA FWD 3.8L V6,Bas,4D SPORT,...,9906.0,11657.0,NaN,NaN,3453,80022,CO,6770.0,0,1389
1,32406,0,2009-01-05,MANHEIM,2005,4,FORD,FREESTAR FWD V6 3.9L,SES,4D PASSENGER 3.9L SES,...,5801.0,6949.0,NaN,NaN,22916,80022,CO,6160.0,0,941
2,32407,0,2009-01-05,MANHEIM,2004,5,DODGE,STRATUS 4C 2.4L I4 M,SE,4D SEDAN SE,...,4169.0,5114.0,NaN,NaN,3453,80022,CO,4250.0,0,1155
3,32408,0,2009-01-05,MANHEIM,2006,3,CHEVROLET,TRAILBLAZER EXT 4WD,LS,4D SUV 4.2L,...,10438.0,12158.0,NaN,NaN,22916,80022,CO,8180.0,0,1703
4,32409,0,2009-01-05,MANHEIM,2004,5,FORD,TAURUS 3.0L V6 EFI,SES,4D SEDAN SES DURATEC,...,4139.0,5351.0,NaN,NaN,22916,80022,CO,4900.0,0,825
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24322,56894,0,2009-09-15,MANHEIM,2005,4,CHEVROLET,AVEO 1.6L I4 EFI,LS,4D SEDAN,...,6341.0,7085.0,NaN,NaN,23359,92504,CA,4085.0,0,803
24323,56893,0,2009-09-15,MANHEIM,2008,1,DODGE,AVENGER 4C 2.4L I4 S,SE,4D SEDAN,...,11871.0,12554.0,NaN,NaN,23359,92504,CA,7555.0,0,1020
24324,56892,0,2009-09-15,MANHEIM,2008,1,DODGE,AVENGER 4C 2.4L I4 S,SE,4D SEDAN,...,9753.0,10571.0,NaN,NaN,23359,92504,CA,7555.0,0,1020
24325,15848,0,2009-09-15,MANHEIM,2006,3,MAZDA,MAZDA6 2.3L I4 MFI /,i,4D SEDAN I,...,8697.0,10288.0,NaN,NaN,20207,77041,TX,7695.0,0,1272


In [ ]:


categorical_cols = ['Auction', 'Make', 'Model', 'Trim', 'SubModel',
                    'Color', 'Transmission', 'WheelType', 'Nationality',
                    'Size', 'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART', 'VNST']

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()

    train_filled = train[col].astype(str).fillna('UNKNOWN')
    le.fit(train_filled)
    encoders[col] = le

    train[col] = le.transform(train_filled)

    # Создаём словарь маппинга
    mapping = {cat: idx for idx, cat in enumerate(le.classes_)}

    # Быстрый map с заполнением неизвестных на -1
    for df in [valid, test]:
        df[col] = df[col].astype(str).fillna('UNKNOWN').map(mapping).fillna(-1).astype(int)

In [ ]:
train.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,32389,0,2009-01-05,1,2007,2,4,527,7,242,...,9906.0,11657.0,1,1,3453,80022,3,6770.0,0,1389
1,32406,0,2009-01-05,1,2005,4,6,296,76,116,...,5801.0,6949.0,1,1,22916,80022,3,6160.0,0,941
2,32407,0,2009-01-05,1,2004,5,5,665,74,199,...,4169.0,5114.0,1,1,3453,80022,3,4250.0,0,1155
3,32408,0,2009-01-05,1,2006,3,3,721,49,299,...,10438.0,12158.0,1,1,22916,80022,3,8180.0,0,1703
4,32409,0,2009-01-05,1,2004,5,6,688,76,213,...,4139.0,5351.0,1,1,22916,80022,3,4900.0,0,825


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Исключаем из признаков: таргет, дату, ID
exclude_cols = ['IsBadBuy', 'PurchDate', 'RefId']
feature_cols = [col for col in train.columns if col not in exclude_cols]

# Заполняем NaN медианой перед нормализацией
imputer = SimpleImputer(strategy='median')
train[feature_cols] = imputer.fit_transform(train[feature_cols])
valid[feature_cols] = imputer.transform(valid[feature_cols])
test[feature_cols] = imputer.transform(test[feature_cols])

# Нормализация 
scaler = StandardScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
valid[feature_cols] = scaler.transform(valid[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

# Разделение на X и y
X_train = train[feature_cols]
y_train = train['IsBadBuy']

X_valid = valid[feature_cols]
y_valid = valid['IsBadBuy']

X_test = test[feature_cols]
y_test = test['IsBadBuy']



## 2 Создание класса

In [ ]:
import numpy as np

class MLP:
    def __init__(self, n_hidden=100, activation='relu', lr=0.01, batch_size=32, epochs=100, optimizer='sgd'):
        self.n_hidden = n_hidden
        self.activation_name = activation
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.optimizer = optimizer

        self.W1 = None
        self.b1 = None
        self.W2 = None
        self.b2 = None

        # Для Adam
        self.m_W1 = self.v_W1 = self.m_b1 = self.v_b1 = None
        self.m_W2 = self.v_W2 = self.m_b2 = self.v_b2 = None
        self.t = 0

    def _init_weights(self, n_features):
        np.random.seed(42)
   
        self.W1 = np.random.randn(n_features, self.n_hidden) * np.sqrt(2.0 / n_features)
        self.b1 = np.zeros((1, self.n_hidden))
        self.W2 = np.random.randn(self.n_hidden, 2) * np.sqrt(2.0 / self.n_hidden)
        self.b2 = np.zeros((1, 2))

        if self.optimizer == 'adam':
            self.m_W1 = np.zeros_like(self.W1)
            self.v_W1 = np.zeros_like(self.W1)
            self.m_b1 = np.zeros_like(self.b1)
            self.v_b1 = np.zeros_like(self.b1)
            self.m_W2 = np.zeros_like(self.W2)
            self.v_W2 = np.zeros_like(self.W2)
            self.m_b2 = np.zeros_like(self.b2)
            self.v_b2 = np.zeros_like(self.b2)
            self.t = 0

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def _sigmoid_derivative(self, x):
        s = self._sigmoid(x)
        return s * (1 - s)

    def _relu(self, x):
        return np.maximum(0, x)

    def _relu_derivative(self, x):
        return (x > 0).astype(float)

    def _get_activation(self, x):
        if self.activation_name == 'sigmoid':
            return self._sigmoid(x)
        elif self.activation_name == 'relu':
            return self._relu(x)

    def _get_activation_derivative(self, x):
        if self.activation_name == 'sigmoid':
            return self._sigmoid_derivative(x)
        elif self.activation_name == 'relu':
            return self._relu_derivative(x)

    def _softmax(self, x):
        # Clip входов чтобы не было переполнения
        x = np.clip(x, -500, 500)
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / (np.sum(exp_x, axis=1, keepdims=True) + 1e-15)
    def _cosine(self, x):
        return np.cos(x)

    def _cosine_derivative(self, x):
        return -np.sin(x)

    def _get_activation(self, x):
        if self.activation_name == 'sigmoid':
            return self._sigmoid(x)
        elif self.activation_name == 'relu':
            return self._relu(x)
        elif self.activation_name == 'cosine':
            return self._cosine(x)

    def _get_activation_derivative(self, x):
        if self.activation_name == 'sigmoid':
            return self._sigmoid_derivative(x)
        elif self.activation_name == 'relu':
            return self._relu_derivative(x)
        elif self.activation_name == 'cosine':
            return self._cosine_derivative(x)
    def _forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self._get_activation(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self._softmax(self.z2)
        return self.a2

    def _compute_loss(self, y_true, y_pred):
        n = len(y_true)
        y_onehot = np.zeros((n, 2))
        y_onehot[np.arange(n), y_true.astype(int)] = 1
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        loss = -np.mean(np.sum(y_onehot * np.log(y_pred), axis=1))
        return loss

    def _backward(self, X, y_true):
        n = len(y_true)
        y_onehot = np.zeros((n, 2))
        y_onehot[np.arange(n), y_true.astype(int)] = 1

        # Градиент выходного слоя
        dz2 = self.a2 - y_onehot
        dW2 = self.a1.T @ dz2 / n
        db2 = np.sum(dz2, axis=0, keepdims=True) / n

        # Градиент скрытого слоя
        da1 = dz2 @ self.W2.T
        dz1 = da1 * self._get_activation_derivative(self.z1)
        dW1 = X.T @ dz1 / n
        db1 = np.sum(dz1, axis=0, keepdims=True) / n

        # Gradient clipping 
        max_grad = 5.0
        dW1 = np.clip(dW1, -max_grad, max_grad)
        db1 = np.clip(db1, -max_grad, max_grad)
        dW2 = np.clip(dW2, -max_grad, max_grad)
        db2 = np.clip(db2, -max_grad, max_grad)

        return dW1, db1, dW2, db2

    def _update_weights(self, dW1, db1, dW2, db2):
        if self.optimizer == 'sgd':
            self.W1 -= self.lr * dW1
            self.b1 -= self.lr * db1
            self.W2 -= self.lr * dW2
            self.b2 -= self.lr * db2

        elif self.optimizer == 'adam':
            self.t += 1
            beta1, beta2, eps = 0.9, 0.999, 1e-8

            # Обновляем каждый параметр отдельно
            self.m_W1 = beta1 * self.m_W1 + (1 - beta1) * dW1
            self.v_W1 = beta2 * self.v_W1 + (1 - beta2) * (dW1 ** 2)
            m_hat_W1 = self.m_W1 / (1 - beta1 ** self.t)
            v_hat_W1 = self.v_W1 / (1 - beta2 ** self.t)
            self.W1 -= self.lr * m_hat_W1 / (np.sqrt(v_hat_W1) + eps)

            self.m_b1 = beta1 * self.m_b1 + (1 - beta1) * db1
            self.v_b1 = beta2 * self.v_b1 + (1 - beta2) * (db1 ** 2)
            m_hat_b1 = self.m_b1 / (1 - beta1 ** self.t)
            v_hat_b1 = self.v_b1 / (1 - beta2 ** self.t)
            self.b1 -= self.lr * m_hat_b1 / (np.sqrt(v_hat_b1) + eps)

            self.m_W2 = beta1 * self.m_W2 + (1 - beta1) * dW2
            self.v_W2 = beta2 * self.v_W2 + (1 - beta2) * (dW2 ** 2)
            m_hat_W2 = self.m_W2 / (1 - beta1 ** self.t)
            v_hat_W2 = self.v_W2 / (1 - beta2 ** self.t)
            self.W2 -= self.lr * m_hat_W2 / (np.sqrt(v_hat_W2) + eps)

            self.m_b2 = beta1 * self.m_b2 + (1 - beta1) * db2
            self.v_b2 = beta2 * self.v_b2 + (1 - beta2) * (db2 ** 2)
            m_hat_b2 = self.m_b2 / (1 - beta1 ** self.t)
            v_hat_b2 = self.v_b2 / (1 - beta2 ** self.t)
            self.b2 -= self.lr * m_hat_b2 / (np.sqrt(v_hat_b2) + eps)

    def fit(self, X, y, X_val=None, y_val=None):
        n_samples, n_features = X.shape
        self._init_weights(n_features)

        for epoch in range(self.epochs):
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i:i+self.batch_size]
                y_batch = y_shuffled[i:i+self.batch_size]

                self._forward(X_batch)
                dW1, db1, dW2, db2 = self._backward(X_batch, y_batch)
                self._update_weights(dW1, db1, dW2, db2)

            if (epoch + 1) % 10 == 0:
                y_pred_train = self._forward(X)
                train_loss = self._compute_loss(y, y_pred_train)
                if X_val is not None:
                    y_pred_val = self._forward(X_val)
                    val_loss = self._compute_loss(y_val, y_pred_val)
                    print(f"Epoch {epoch+1}/{self.epochs}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
                else:
                    print(f"Epoch {epoch+1}/{self.epochs}: train_loss={train_loss:.4f}")

        return self

    def predict_proba(self, X):
        return self._forward(X)

    def predict(self, X):
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)

In [ ]:
model = MLP(n_hidden=50, activation='relu', lr=0.001, batch_size=32, epochs=40, optimizer='adam')
model.fit(X_train.values, y_train.values, X_valid.values, y_valid.values)

Epoch 10/40: train_loss=0.2862, val_loss=0.3569
Epoch 20/40: train_loss=0.2812, val_loss=0.3545
Epoch 30/40: train_loss=0.2766, val_loss=0.3519
Epoch 40/40: train_loss=0.2751, val_loss=0.3531


## 3 Проверяем Gini score > 0,15


In [ ]:
y_proba = model.predict_proba(X_valid.values)
roc_auc = roc_auc_score(y_valid.values, y_proba[:, 1])
gini = 2 * roc_auc - 1

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Gini: {gini:.4f}")

ROC-AUC: 0.7253
Gini: 0.4507


## 4 Sklearn модели

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

# Sklearn MLP
sklearn_mlp = MLPClassifier(
    hidden_layer_sizes=(50,),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=40,
    batch_size=32,
    random_state=42
)

sklearn_mlp.fit(X_train.values, y_train.values)

# Предсказания
y_proba_sklearn = sklearn_mlp.predict_proba(X_valid.values)
gini_sklearn = 2 * roc_auc_score(y_valid.values, y_proba_sklearn[:, 1]) - 1



/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
print(f"Sklearn MLP Gini: {gini_sklearn:.4f}")
print(f"Твой MLP Gini:    {gini:.4f}")


Sklearn MLP Gini: 0.4588
Твой MLP Gini:    0.4507


## 5 Различные параметры для моделей

In [ ]:
from sklearn.metrics import roc_auc_score

activations = ['relu', 'sigmoid', 'cosine']
results = []

for act in activations:
    model = MLP(n_hidden=50, activation=act, lr=0.001, batch_size=32, epochs=40, optimizer='adam')
    model.fit(X_train.values, y_train.values, X_valid.values, y_valid.values)

    y_proba = model.predict_proba(X_valid.values)
    gini = 2 * roc_auc_score(y_valid.values, y_proba[:, 1]) - 1
    results.append((act, gini))
    print(f"{act:10s} → Gini: {gini:.4f}")

# Лучшая функция
best = max(results, key=lambda x: x[1])
print(f"\n Лучшая: {best[0]} (Gini = {best[1]:.4f})")

Epoch 10/40: train_loss=0.2862, val_loss=0.3569
Epoch 20/40: train_loss=0.2812, val_loss=0.3545
Epoch 30/40: train_loss=0.2766, val_loss=0.3519
Epoch 40/40: train_loss=0.2751, val_loss=0.3531
relu       → Gini: 0.4507
Epoch 10/40: train_loss=0.2936, val_loss=0.3436
Epoch 20/40: train_loss=0.2899, val_loss=0.3400
Epoch 30/40: train_loss=0.2859, val_loss=0.3409
Epoch 40/40: train_loss=0.2827, val_loss=0.3408
sigmoid    → Gini: 0.4567
Epoch 10/40: train_loss=0.2824, val_loss=0.3437
Epoch 20/40: train_loss=0.2720, val_loss=0.3451
Epoch 30/40: train_loss=0.2635, val_loss=0.3545
Epoch 40/40: train_loss=0.2597, val_loss=0.3600
cosine     → Gini: 0.4137

 Лучшая: sigmoid (Gini = 0.4567)


In [ ]:
sklearn_results = []
activations = ['relu', 'logistic', 'tanh']
for act in activations:
    model = MLPClassifier(
        hidden_layer_sizes=(50,),
        activation=act,
        solver='adam',
        learning_rate_init=0.001,
        max_iter=40,
        batch_size=32,
        random_state=42
    )
    model.fit(X_train.values, y_train.values)

    y_proba = model.predict_proba(X_valid.values)
    gini = 2 * roc_auc_score(y_valid.values, y_proba[:, 1]) - 1
    sklearn_results.append((act, gini))
    print(f"{act:10s} → Gini: {gini:.4f}")

best_sklearn = max(sklearn_results, key=lambda x: x[1])
print(f" Лучшая: {best_sklearn[0]} (Gini = {best_sklearn[1]:.4f})")


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


relu       → Gini: 0.4588


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


logistic   → Gini: 0.4688
tanh       → Gini: 0.4198
 Лучшая: logistic (Gini = 0.4688)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


## 6 Модель с 1 скрытым слоем

In [ ]:
activations = ['relu', 'sigmoid', 'tanh']
keras_results = []

for act in activations:
    keras_model = keras.Sequential([
        keras.layers.Dense(50, activation=act, input_shape=(X_train.shape[1],)),
        keras.layers.Dense(2, activation='softmax')
    ])

    keras_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    keras_model.fit(
        X_train.values, y_train.values,
        validation_data=(X_valid.values, y_valid.values),
        epochs=40,
        batch_size=32,
        verbose=0
    )

    y_proba = keras_model.predict(X_valid.values, verbose=0)
    gini = 2 * roc_auc_score(y_valid.values, y_proba[:, 1]) - 1
    keras_results.append((act, gini))
    print(f"{act:10s} → Gini: {gini:.4f}")

best_keras = max(keras_results, key=lambda x: x[1])



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


relu       → Gini: 0.4477


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


sigmoid    → Gini: 0.4609


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


tanh       → Gini: 0.4287


In [ ]:
print(f" Собственная реализация: {best[0]} (Gini = {best[1]:.4f})")
print(f" Лучшая Sklearn: {best_sklearn[0]} (Gini = {best_sklearn[1]:.4f})")
print(f" Лучшая keras с скрытым слоем: {best_keras[0]} (Gini = {best_keras[1]:.4f})")



 Собственная реализация: sigmoid (Gini = 0.4567)
 Лучшая Sklearn: logistic (Gini = 0.4688)
 Лучшая keras с скрытым слоем: sigmoid (Gini = 0.4609)


## 7 Лучсшая модель на обучающей, тестовой и валидационной выборке

In [ ]:
best_model = MLPClassifier(
    hidden_layer_sizes=(50,),
    activation='logistic',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=40,
    batch_size=32,
    random_state=42
)

best_model.fit(X_train.values, y_train.values)

# Предсказания на всех наборах
y_proba_train = best_model.predict_proba(X_train.values)
y_proba_valid = best_model.predict_proba(X_valid.values)
y_proba_test = best_model.predict_proba(X_test.values)

# Gini для каждого набора
gini_train = 2 * roc_auc_score(y_train.values, y_proba_train[:, 1]) - 1
gini_valid = 2 * roc_auc_score(y_valid.values, y_proba_valid[:, 1]) - 1
gini_test = 2 * roc_auc_score(y_test.values, y_proba_test[:, 1]) - 1


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
print(f"Train Gini:  {gini_train:.4f}")
print(f"Valid Gini:  {gini_valid:.4f}")
print(f"Test Gini:   {gini_test:.4f}")

Train Gini:  0.5406
Valid Gini:  0.4688
Test Gini:   0.3067
